# CHC (CHIRPS) — catalog explorer (no network)

The `chc` backend reaches the full Climate Hazards Center product line — CHIRPS / CHIRP precipitation, CHIRTS temperature & humidity, CHIRPS-GEFS forecasts, CHPclim climatology, WBGT, and the SPI / SPEI drought indices — over anonymous FTP. This notebook explores the **bundled catalog offline**: no FTP access, so it runs deterministically at docs-build time.

## Setup

Load the bundled `Catalog`. It reads `chc_data_catalog.yaml` from package data, so no network call is made. The counts below summarise the curated `datasets` map, the informational `available_datasets` index, and the named region sub-grids.

In [ ]:
from earthlens.chc import Catalog

cat = Catalog()
print(f'datasets in catalog: {len(cat.datasets)}')
print(f'available_datasets index: {len(cat.available_datasets)}')
print(f'named regions: {len(cat.available_regions)}')

## Datasets by temporal resolution

Each catalog entry knows its native cadence (daily, monthly, dekadal, pentadal, …) and the pandas frequency the backend builds its date axis from. Counting entries per cadence shows where the product line is concentrated.

In [ ]:
from collections import Counter

by_res = Counter(d.temporal_resolution for d in cat.datasets.values())
for res, n in sorted(by_res.items(), key=lambda kv: -kv[1]):
    print(f'{n:>3}  {res}')

## Inspect one dataset

`global-daily` is the headline CHIRPS-2.0 product — 0.05° global daily rainfall back to 1981. Pull its entry out of the `datasets` map and print the grid, date, and format metadata the backend uses to shape a download.

In [ ]:
d = cat.datasets['global-daily']
print('region            :', d.region)
print(
    'temporal_resolution:', d.temporal_resolution, '(pandas freq', d.pandas_freq + ')'
)
print('spatial_resolution :', d.spatial_resolution, 'deg')
print('lat boundaries     :', d.lat_boundaries)
print('lon boundaries     :', d.lon_boundaries)
print('start_date         :', d.start_date)
print('formats            :', d.formats)
print('variables          :', list(d.variables))

## Look up a variable

Every dataset exposes a `variables` map of `Variable` metadata — name, units (via type), and a human description. Here we read the `precipitation` variable off `global-daily`.

In [ ]:
var = cat.datasets['global-daily'].variables['precipitation']
print('name       :', var.name)
print('type        :', var.types)
print('description :', var.description)

## Named regions

CHC ships several regional sub-grids; the catalog expands per-region dataset variants from this block. Each entry maps a region name to its latitude / longitude bounding box.

In [ ]:
for name, box in cat.available_regions.items():
    print(f'{name:<28} {box}')